# ANARCI + BLASTp vs MMseqs2 benchmark

This standalone Google Colab notebook performs one fair experiment from beginning to end:

1. Upload a protein FASTA containing all chains from the TCR–pMHC structures.
2. Use ANARCI to recognize TCR alpha (`A`) and beta (`B`) variable domains and assign IMGT numbering.
3. Extract the complete numbered variable-domain sequences and CDR1/2/3-IMGT.
4. Search the **same alpha and beta variable-domain FASTAs** with BLASTp and MMseqs2.
5. Measure database-build time, search time, and peak memory.
6. Create an independent ground truth with exhaustive exact Smith–Waterman alignments.
7. Compare precision, recall and F1 at multiple identity thresholds.

BLASTp is compared with MMseqs2 `search`, not `easy-cluster`, because BLAST is a search program and does not itself cluster sequences. ANARCI is the annotation stage before either search program; it is not included in their timed search comparison.


## 0. Start with a clean Colab runtime

Use **Runtime → Disconnect and delete runtime**, reconnect, and run every cell in order. This avoids mixing incompatible ANARCI installations. Do not separately install `anarci-mhc`, clone another ANARCI repository, or build models under `~/.anarci` in this runtime.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y hmmer ncbi-blast+ time
%pip install -q "biopython==1.85" "ANARCI==2026.2.13.2" pandas matplotlib tqdm parasail


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package time.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../time_1.9-0.1build2_amd64.deb ...
Unpacking time (1.9-0.1build2) ...
Selecting previously unselected package libdivsufsort3:amd64.
Preparing to unpack .../libdivsufsort3_2.0.1-5_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-5) ...
Selecting previously unselected package ncbi-data.
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_amd64.deb ...
Unpacking ncbi-blast+ (2.12.0+ds-3build1) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.3.2+dfsg-1_amd64.deb 

In [ ]:
%%bash
set -euo pipefail
cd /content
if grep -qm1 avx2 /proc/cpuinfo; then
  mmseqs_build="avx2"
else
  mmseqs_build="sse41"
fi
wget -q -O mmseqs-linux.tar.gz "https://mmseqs.com/latest/mmseqs-linux-${mmseqs_build}.tar.gz"
tar -xzf mmseqs-linux.tar.gz
ln -sf /content/mmseqs/bin/mmseqs /usr/local/bin/mmseqs
mmseqs version


17b688d21dda57fc5f5b7286ecba7ec003d4717f


## 1. Validate the software

ANARCI internally uses HMMER. BLASTp and MMseqs2 are then run independently on ANARCI's accepted variable domains.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
import importlib.metadata
import math
import random
import shutil
import subprocess
import time

import anarci
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import parasail
from Bio import SeqIO
from IPython.display import display
from tqdm.auto import tqdm
from anarci.anarci import HMM_path

HMM_DATABASE = Path(HMM_path) / "ALL.hmm"
assert shutil.which("hmmscan"), "HMMER hmmscan was not found"
assert shutil.which("blastp"), "NCBI blastp was not found"
assert shutil.which("makeblastdb"), "NCBI makeblastdb was not found"
assert shutil.which("mmseqs"), "MMseqs2 was not found"
assert Path("/usr/bin/time").exists(), "/usr/bin/time was not found"
assert HMM_DATABASE.is_file(), f"ANARCI model database was not found: {HMM_DATABASE}"

print("ANARCI:", anarci.__version__)
print("ANARCI HMM database:", HMM_DATABASE)
print(subprocess.check_output(["blastp", "-version"], text=True).splitlines()[0])
print("MMseqs2:", subprocess.check_output(["mmseqs", "version"], text=True).strip())
print("Parasail:", importlib.metadata.version("parasail"))


ANARCI: 2026.2.13.2
ANARCI HMM database: /usr/local/lib/python3.12/dist-packages/anarci/dat/HMMs/ALL.hmm
blastp: 2.12.0+
MMseqs2: 17b688d21dda57fc5f5b7286ecba7ec003d4717f
Parasail: 1.3.4


## 2. Benchmark configuration

The default benchmark tests high, medium and lower similarity. Change these values before running the analysis if your research question uses different thresholds.

- `IDENTITY_THRESHOLDS = [50, 70, 90]` evaluates several difficulty levels.
- `MIN_COVERAGE = 80` requires both sequences to have at least 80% coverage.
- `MMSEQS_SENSITIVITIES` produces a runtime–accuracy curve.
- `MAX_SEQUENCES_PER_CHAIN` prevents an accidental enormous exact all-pairs calculation. Your current dataset should remain below the default.


In [ ]:
THREADS = 2
N_REPEATS = 3
RANDOM_SEED = 20260814
IDENTITY_THRESHOLDS = [50.0, 70.0, 90.0]
MIN_COVERAGE = 80.0
SEARCH_EVALUE = 10.0
MAX_HITS_PER_QUERY = 100000
MMSEQS_SENSITIVITIES = [4.0, 5.7, 7.5]
MAX_SEQUENCES_PER_CHAIN = 1000

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = Path("/content") / f"ANARCI_BLAST_MMseqs2_benchmark_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print("Output directory:", OUTPUT_DIR)


Output directory: /content/ANARCI_BLAST_MMseqs2_benchmark_20260817T142256262701Z


## 3. Upload the protein FASTA

Upload the FASTA containing all available chains. Short peptide chains, MHC chains and beta-2 microglobulin may be present; ANARCI will only retain recognized TCR alpha and beta variable domains.


In [ ]:
from google.colab import files

uploaded = files.upload()
fasta_names = [name for name in uploaded if name.lower().endswith((".fa", ".faa", ".fasta", ".fas"))]
if len(fasta_names) != 1:
    raise ValueError(f"Upload exactly one protein FASTA file; found {fasta_names}.")
FASTA_PATH = Path(fasta_names[0])
print("Input FASTA:", FASTA_PATH)


Saving all_sequences.fasta to all_sequences.fasta
Input FASTA: all_sequences.fasta


## 4. Validate sequences before ANARCI

Sequences shorter than 70 residues and sequences containing nonstandard amino-acid symbols are logged rather than silently changed.


In [ ]:
VALID_AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWY")
records = list(SeqIO.parse(str(FASTA_PATH), "fasta"))
if not records:
    raise ValueError(f"No FASTA records were found in {FASTA_PATH}.")

input_rows = []
candidate_rows = []
failure_rows = []

for source_index, record in enumerate(records):
    sequence = str(record.seq).upper().replace("*", "")
    invalid_letters = sorted(set(sequence) - VALID_AMINO_ACIDS)
    item = {
        "analysis_id": f"seq{source_index:06d}",
        "source_index": source_index,
        "source_id": record.id,
        "description": record.description,
        "sequence": sequence,
        "input_length": len(sequence),
        "invalid_letters": "".join(invalid_letters),
    }
    input_rows.append(item)
    if len(sequence) < 70:
        failure_rows.append({**item, "failure_stage": "prefilter", "failure_reason": "sequence_shorter_than_70_aa"})
    elif invalid_letters:
        failure_rows.append({**item, "failure_stage": "prefilter", "failure_reason": "nonstandard_amino_acid_letter"})
    else:
        candidate_rows.append(item)

input_df = pd.DataFrame(input_rows)
print("Total FASTA records:", len(input_rows))
print("Candidates sent to ANARCI:", len(candidate_rows))
print("Prefilter failures:", len(failure_rows))
display(input_df[["analysis_id", "source_id", "input_length", "invalid_letters"]].head())


Total FASTA records: 1861
Candidates sent to ANARCI: 1511
Prefilter failures: 350


,analysis_id,source_id,input_length,invalid_letters
0,seq000000,1AO7_2|Chain,100,
1,seq000001,1AO7_3|Chain,9,
2,seq000002,1AO7_4|Chain,204,
3,seq000003,1AO7_5|Chain,245,
4,seq000004,1AO7_1|Chain,275,


## 5. Run ANARCI and extract IMGT regions

ANARCI scans each candidate against profile HMMs, retains alpha/beta variable domains with `allow={"A", "B"}`, and assigns IMGT numbers. Our code then extracts occupied positions 27–38, 56–65 and 105–117.


In [ ]:
from anarci import anarci as run_anarci

if not candidate_rows:
    raise ValueError("No valid sequences remained after prefiltering.")

sequence_pairs = [(row["analysis_id"], row["sequence"]) for row in candidate_rows]
anarci_started = time.perf_counter()
numbered_results, detail_results, hit_tables = run_anarci(
    sequence_pairs,
    scheme="imgt",
    allow={"A", "B"},
    output=False,
    ncpu=THREADS,
)
anarci_wall_seconds = time.perf_counter() - anarci_started
assert len(numbered_results) == len(candidate_rows)
assert len(detail_results) == len(candidate_rows)
print("ANARCI processed", len(candidate_rows), f"candidate sequences in {anarci_wall_seconds:.2f} seconds.")


Limiting hmmer search to species ['human', 'mouse'] was requested but hits did not achieve a high enough bitscore. Reverting to using any species
ANARCI processed 1511 candidate sequences in 21.21 seconds.


In [ ]:
IMGT_REGIONS = {
    "FR1_IMGT": (1, 26),
    "CDR1_IMGT": (27, 38),
    "FR2_IMGT": (39, 55),
    "CDR2_IMGT": (56, 65),
    "FR3_IMGT": (66, 104),
    "CDR3_IMGT": (105, 117),
    "FR4_IMGT": (118, 128),
}

def occupied_sequence(numbered_domain):
    return "".join(aa for _, aa in numbered_domain if aa != "-")

def region_entries(numbered_domain, start, end):
    return [
        ((int(position), str(insertion).strip()), aa)
        for ((position, insertion), aa) in numbered_domain
        if start <= int(position) <= end and aa != "-"
    ]

def region_sequence(numbered_domain, start, end):
    return "".join(aa for _, aa in region_entries(numbered_domain, start, end))

def region_numbering(numbered_domain, start, end):
    return " ".join(
        f"{position}{insertion}={aa}"
        for ((position, insertion), aa) in region_entries(numbered_domain, start, end)
    )


In [ ]:
result_rows = []

for item, numbered_entry, details_entry in zip(candidate_rows, numbered_results, detail_results):
    if numbered_entry is None or details_entry is None:
        failure_rows.append({**item, "failure_stage": "ANARCI", "failure_reason": "no_alpha_or_beta_TCR_hit"})
        continue
    for domain_index, (domain, info) in enumerate(zip(numbered_entry, details_entry)):
        numbered_domain, sequence_start, sequence_end = domain
        chain_type = info["chain_type"]
        if chain_type not in {"A", "B"}:
            continue
        row = {
            "benchmark_id": f"{item['analysis_id']}_D{domain_index}_{chain_type}",
            "analysis_id": item["analysis_id"],
            "source_id": item["source_id"],
            "description": item["description"],
            "domain_index": domain_index,
            "chain_type": chain_type,
            "species_model": info.get("species"),
            "anarci_bitscore": info.get("bitscore"),
            "anarci_evalue": info.get("evalue"),
            "sequence_start": sequence_start,
            "sequence_end": sequence_end,
            "input_length": item["input_length"],
            "variable_domain_sequence": occupied_sequence(numbered_domain),
        }
        for region_name, (start, end) in IMGT_REGIONS.items():
            seq = region_sequence(numbered_domain, start, end)
            row[region_name] = seq
            row[f"{region_name}_length"] = len(seq)
            if region_name.startswith("CDR"):
                row[f"{region_name}_numbered"] = region_numbering(numbered_domain, start, end)
        result_rows.append(row)

cdr_df = pd.DataFrame(result_rows)
failure_df = pd.DataFrame(failure_rows)
if cdr_df.empty:
    raise RuntimeError("ANARCI found no TCR alpha/beta variable domains.")
if cdr_df.benchmark_id.duplicated().any():
    raise RuntimeError("Benchmark identifiers are not unique.")

print("Accepted variable domains:", len(cdr_df))
display(cdr_df.chain_type.value_counts().rename(index={"A": "alpha", "B": "beta"}))
display(cdr_df[["benchmark_id", "source_id", "chain_type", "anarci_bitscore", "CDR1_IMGT", "CDR2_IMGT", "CDR3_IMGT"]].head())


Accepted variable domains: 732


,count
chain_type,
beta,378
alpha,354


,benchmark_id,source_id,chain_type,anarci_bitscore,CDR1_IMGT,CDR2_IMGT,CDR3_IMGT
0,seq000002_D0_A,1AO7_4|Chain,A,129.9,DRGSQS,IYSNGD,AVTTDSWGKLQ
1,seq000003_D0_B,1AO7_5|Chain,B,143.1,MNHEY,SVGAGI,ASRPGLAGGRPEQY
2,seq000007_D0_B,1BD2_5|Chain,B,148.9,MNHEY,SVGAGI,ASSYPGGGFYEQY
3,seq000014_D0_B,1FO0_5|Chain,B,106.3,NSQYPW,LRSPGD,TCSADRVGNTLY
4,seq000018_D0_B,1G6R_2|Chains,B,135.1,NNHNN,SYGAGS,ASGGGGTLY


## 6. Create identical benchmark inputs for both search tools

Alpha and beta are benchmarked separately. If a chain set exceeds `MAX_SEQUENCES_PER_CHAIN`, a reproducible random subset is used for both tools and for the exact ground truth.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
benchmark_parts = []
for chain_type in ["A", "B"]:
    subset = cdr_df[cdr_df.chain_type == chain_type].copy()
    if len(subset) > MAX_SEQUENCES_PER_CHAIN:
        chosen = rng.choice(subset.index.to_numpy(), size=MAX_SEQUENCES_PER_CHAIN, replace=False)
        subset = subset.loc[chosen].copy()
        print(f"Chain {chain_type}: sampled {len(subset)} domains for the benchmark.")
    benchmark_parts.append(subset)

benchmark_df = pd.concat(benchmark_parts, ignore_index=True)
benchmark_fastas = {}
for chain_type, chain_name in {"A": "alpha", "B": "beta"}.items():
    subset = benchmark_df[benchmark_df.chain_type == chain_type].sort_values("benchmark_id")
    if len(subset) < 2:
        print(f"Skipping {chain_name}: only {len(subset)} accepted domain(s).")
        continue
    fasta_path = OUTPUT_DIR / f"TCR_{chain_name}_variable_domains.fasta"
    with fasta_path.open("w") as handle:
        for row in subset.itertuples(index=False):
            handle.write(f">{row.benchmark_id} source_id={row.source_id} chain={row.chain_type}\n{row.variable_domain_sequence}\n")
    benchmark_fastas[chain_type] = fasta_path
    print(chain_name, "benchmark sequences:", len(subset), "|", fasta_path)

if not benchmark_fastas:
    raise RuntimeError("Neither chain type had at least two sequences for an all-against-all benchmark.")


alpha benchmark sequences: 354 | /content/ANARCI_BLAST_MMseqs2_benchmark_20260817T142256262701Z/TCR_alpha_variable_domains.fasta
beta benchmark sequences: 378 | /content/ANARCI_BLAST_MMseqs2_benchmark_20260817T142256262701Z/TCR_beta_variable_domains.fasta


## 7. Runtime benchmark

Fairness rules used here:

- Same FASTA, same chain split, same number of CPU threads and same permissive E-value.
- Self hits are removed later.
- Hit limits are set above the dataset size so default truncation does not hide neighbors.
- Database construction and search are timed separately.
- MMseqs2 output conversion is included in its search time because BLAST directly writes a TSV.
- Search order alternates across repeats to reduce warm-cache order bias.
- We report median search time over repeated runs.


In [ ]:
def parse_gnu_time(time_path):
    peak_kb = math.nan
    for line in Path(time_path).read_text().splitlines():
        if "Maximum resident set size (kbytes)" in line:
            peak_kb = float(line.rsplit(":", 1)[1].strip())
    return peak_kb / 1024.0

def run_timed(command, label):
    time_path = OUTPUT_DIR / f"{label}.gnu_time.txt"
    wrapped = ["/usr/bin/time", "-v", "-o", str(time_path)] + [str(x) for x in command]
    started = time.perf_counter()
    completed = subprocess.run(wrapped, text=True, capture_output=True)
    wall_seconds = time.perf_counter() - started
    if completed.returncode != 0:
        print("Command failed:", " ".join(map(str, command)))
        print(completed.stdout[-4000:])
        print(completed.stderr[-4000:])
        raise subprocess.CalledProcessError(completed.returncode, command)
    return {
        "wall_seconds": wall_seconds,
        "peak_memory_mb": parse_gnu_time(time_path),
        "time_file": str(time_path),
    }

runtime_rows = []
database_paths = {}
search_output_paths = {}


In [ ]:
for chain_type, fasta_path in benchmark_fastas.items():
    chain_name = {"A": "alpha", "B": "beta"}[chain_type]
    chain_dir = OUTPUT_DIR / chain_name
    chain_dir.mkdir(parents=True, exist_ok=True)

    blast_db = chain_dir / "blast_db"
    blast_build = run_timed(
        ["makeblastdb", "-in", fasta_path, "-dbtype", "prot", "-out", blast_db],
        f"{chain_name}_blast_build",
    )
    runtime_rows.append({
        "chain_type": chain_type, "tool": "BLASTp", "setting": "standard",
        "stage": "database_build", "repeat": 0, **blast_build,
    })

    mmseqs_db = chain_dir / "mmseqs_db"
    mmseqs_build = run_timed(
        ["mmseqs", "createdb", fasta_path, mmseqs_db],
        f"{chain_name}_mmseqs_build",
    )
    runtime_rows.append({
        "chain_type": chain_type, "tool": "MMseqs2", "setting": "shared_database",
        "stage": "database_build", "repeat": 0, **mmseqs_build,
    })
    database_paths[chain_type] = {"blast": blast_db, "mmseqs": mmseqs_db}

    for repeat in range(1, N_REPEATS + 1):
        tool_order = ["blast", "mmseqs"] if repeat % 2 else ["mmseqs", "blast"]
        for tool_name in tool_order:
            if tool_name == "blast":
                blast_output = chain_dir / f"blast_repeat{repeat}.tsv"
                timing = run_timed([
                    "blastp", "-query", fasta_path, "-db", blast_db,
                    "-out", blast_output,
                    "-outfmt", "6 qseqid sseqid pident length nident qlen slen qstart qend sstart send evalue bitscore",
                    "-evalue", SEARCH_EVALUE,
                    "-max_target_seqs", MAX_HITS_PER_QUERY,
                    "-num_threads", THREADS,
                ], f"{chain_name}_blast_search_repeat{repeat}")
                runtime_rows.append({
                    "chain_type": chain_type, "tool": "BLASTp", "setting": "standard",
                    "stage": "search_and_output", "repeat": repeat, **timing,
                })
                search_output_paths[(chain_type, "BLASTp", "standard")] = blast_output
            else:
                for sensitivity in MMSEQS_SENSITIVITIES:
                    setting = f"s={sensitivity:g}"
                    tag = str(sensitivity).replace(".", "p")
                    result_db = chain_dir / f"mmseqs_result_s{tag}_repeat{repeat}"
                    tmp_dir = chain_dir / f"tmp_s{tag}_repeat{repeat}"
                    mmseqs_output = chain_dir / f"mmseqs_s{tag}_repeat{repeat}.tsv"
                    search_timing = run_timed([
                        "mmseqs", "search", mmseqs_db, mmseqs_db, result_db, tmp_dir,
                        "--threads", THREADS, "-s", sensitivity,
                        "-e", SEARCH_EVALUE, "--max-seqs", MAX_HITS_PER_QUERY, "-a",
                    ], f"{chain_name}_mmseqs_s{tag}_search_repeat{repeat}")
                    conversion_timing = run_timed([
                        "mmseqs", "convertalis", mmseqs_db, mmseqs_db, result_db, mmseqs_output,
                        "--format-output", "query,target,pident,alnlen,nident,qlen,tlen,qstart,qend,tstart,tend,evalue,bits",
                    ], f"{chain_name}_mmseqs_s{tag}_convert_repeat{repeat}")
                    runtime_rows.append({
                        "chain_type": chain_type, "tool": "MMseqs2", "setting": setting,
                        "stage": "search_and_output", "repeat": repeat,
                        "wall_seconds": search_timing["wall_seconds"] + conversion_timing["wall_seconds"],
                        "peak_memory_mb": max(search_timing["peak_memory_mb"], conversion_timing["peak_memory_mb"]),
                        "time_file": search_timing["time_file"] + ";" + conversion_timing["time_file"],
                    })
                    search_output_paths[(chain_type, "MMseqs2", setting)] = mmseqs_output

runtime_df = pd.DataFrame(runtime_rows)
display(runtime_df)


,chain_type,tool,setting,stage,repeat,wall_seconds,peak_memory_mb,time_file
0,A,BLASTp,standard,database_build,0,0.047026,29.000000,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...
1,A,MMseqs2,shared_database,database_build,0,0.012743,6.375000,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...
2,A,BLASTp,standard,search_and_output,1,47.004971,175.496094,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...
3,A,MMseqs2,s=4,search_and_output,1,5.660966,864.875000,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...
4,A,MMseqs2,s=5.7,search_and_output,1,8.935903,865.250000,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...
5,A,MMseqs2,s=7.5,search_and_output,1,9.142554,865.875000,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...
6,A,MMseqs2,s=4,search_and_output,2,7.272081,865.000000,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...
7,A,MMseqs2,s=5.7,search_and_output,2,7.280869,865.359375,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...
8,A,MMseqs2,s=7.5,search_and_output,2,10.588447,865.820312,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...
9,A,BLASTp,standard,search_and_output,2,43.656042,176.761719,/content/ANARCI_BLAST_MMseqs2_benchmark_202608...


## 8. Parse and standardize search results

Both outputs are converted to the same columns. Coverage is recalculated from alignment coordinates rather than relying on tool-specific coverage fields. Self-hits and reverse duplicate pairs are removed, and the best-scoring alignment is kept for each unordered pair.


In [ ]:
SEARCH_COLUMNS = [
    "query", "target", "pident", "alnlen", "nident", "qlen", "tlen",
    "qstart", "qend", "tstart", "tend", "evalue", "bits",
]
NUMERIC_COLUMNS = [c for c in SEARCH_COLUMNS if c not in {"query", "target"}]

def read_and_standardize_search(path):
    path = Path(path)
    if path.stat().st_size == 0:
        return pd.DataFrame(columns=SEARCH_COLUMNS + ["qcov", "tcov", "min_coverage", "pair_a", "pair_b"])
    frame = pd.read_csv(path, sep="\t", names=SEARCH_COLUMNS, header=None)
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.dropna(subset=["query", "target", "pident", "qlen", "tlen", "qstart", "qend", "tstart", "tend", "bits"])
    frame = frame[frame.query.astype(str) != frame.target.astype(str)].copy()
    frame["qcov"] = 100.0 * (frame.qend - frame.qstart).abs().add(1) / frame.qlen
    frame["tcov"] = 100.0 * (frame.tend - frame.tstart).abs().add(1) / frame.tlen
    frame["min_coverage"] = frame[["qcov", "tcov"]].min(axis=1)
    frame["pair_a"] = np.minimum(frame.query.astype(str), frame.target.astype(str))
    frame["pair_b"] = np.maximum(frame.query.astype(str), frame.target.astype(str))
    frame = frame.sort_values("bits", ascending=False).drop_duplicates(["pair_a", "pair_b"], keep="first")
    return frame.reset_index(drop=True)

standardized_results = {}
for key, path in search_output_paths.items():
    standardized_results[key] = read_and_standardize_search(path)
    print(key, "unique non-self pairs:", len(standardized_results[key]))


AttributeError: 'function' object has no attribute 'astype'

## 9. Independent exact-alignment ground truth

Calling BLAST the ground truth would favor BLAST. Instead, this notebook exhaustively aligns every unique pair with Parasail's exact Smith–Waterman implementation using BLOSUM62 and gap penalties 11/1. A true pair at each threshold must satisfy both the exact identity threshold and `MIN_COVERAGE`.

This may be the slowest cell because it evaluates $n(n-1)/2$ pairs per chain. It is deliberately outside the BLAST/MMseqs2 runtime measurement.


In [ ]:
sequence_lookup = benchmark_df.set_index("benchmark_id")["variable_domain_sequence"].to_dict()
truth_tables = {}
truth_runtime_rows = []

def exact_alignment_metrics(pair):
    id_a, id_b = pair
    seq_a, seq_b = sequence_lookup[id_a], sequence_lookup[id_b]
    result = parasail.sw_trace_striped_16(seq_a, seq_b, 11, 1, parasail.blosum62)
    if getattr(result, "saturated", False):
        result = parasail.sw_trace_striped_32(seq_a, seq_b, 11, 1, parasail.blosum62)
    aligned_a = result.traceback.query
    aligned_b = result.traceback.ref
    alignment_length = len(aligned_a)
    matches = sum(a == b and a != "-" for a, b in zip(aligned_a, aligned_b))
    covered_a = sum(a != "-" for a in aligned_a)
    covered_b = sum(b != "-" for b in aligned_b)
    return {
        "pair_a": id_a, "pair_b": id_b,
        "exact_score": result.score,
        "exact_alignment_length": alignment_length,
        "exact_matches": matches,
        "exact_pident": 100.0 * matches / alignment_length if alignment_length else 0.0,
        "exact_qcov": 100.0 * covered_a / len(seq_a),
        "exact_tcov": 100.0 * covered_b / len(seq_b),
        "exact_min_coverage": min(100.0 * covered_a / len(seq_a), 100.0 * covered_b / len(seq_b)),
    }

for chain_type in benchmark_fastas:
    ids = sorted(benchmark_df.loc[benchmark_df.chain_type == chain_type, "benchmark_id"].tolist())
    pairs = list(combinations(ids, 2))
    print(f"Chain {chain_type}: computing {len(pairs):,} exact alignments...")
    started = time.perf_counter()
    rows = [exact_alignment_metrics(pair) for pair in tqdm(pairs)]
    elapsed = time.perf_counter() - started
    truth_tables[chain_type] = pd.DataFrame(rows)
    truth_runtime_rows.append({"chain_type": chain_type, "pair_count": len(pairs), "ground_truth_seconds": elapsed})
    display(truth_tables[chain_type].head())

truth_runtime_df = pd.DataFrame(truth_runtime_rows)
display(truth_runtime_df)


## 10. Precision, recall and F1

For each threshold:

- **True positive:** returned by the search tool and positive in the exact alignment.
- **False positive:** returned as positive but negative in the exact alignment.
- **False negative:** positive in the exact alignment but missed by the tool.
- Precision measures how many reported positives are correct.
- Recall measures how many true positives were recovered.
- F1 balances precision and recall.


In [ ]:
accuracy_rows = []

for chain_type, truth_df in truth_tables.items():
    for identity_threshold in IDENTITY_THRESHOLDS:
        truth_positive_df = truth_df[(truth_df.exact_pident >= identity_threshold) & (truth_df.exact_min_coverage >= MIN_COVERAGE)]
        truth_positive = set(zip(truth_positive_df.pair_a, truth_positive_df.pair_b))
        for (result_chain, tool, setting), search_df in standardized_results.items():
            if result_chain != chain_type:
                continue
            predicted_df = search_df[(search_df.pident >= identity_threshold) & (search_df.min_coverage >= MIN_COVERAGE)]
            predicted_positive = set(zip(predicted_df.pair_a, predicted_df.pair_b))
            true_positive = len(truth_positive & predicted_positive)
            false_positive = len(predicted_positive - truth_positive)
            false_negative = len(truth_positive - predicted_positive)
            precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else math.nan
            recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else math.nan
            f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else math.nan
            accuracy_rows.append({
                "chain_type": chain_type, "tool": tool, "setting": setting,
                "identity_threshold": identity_threshold, "coverage_threshold": MIN_COVERAGE,
                "true_pairs": len(truth_positive), "predicted_pairs": len(predicted_positive),
                "TP": true_positive, "FP": false_positive, "FN": false_negative,
                "precision": precision, "recall": recall, "f1": f1,
            })

accuracy_df = pd.DataFrame(accuracy_rows)
display(accuracy_df.sort_values(["chain_type", "identity_threshold", "f1"], ascending=[True, True, False]))


## 11. Runtime summary and runtime–accuracy tradeoff

`median_search_seconds` is the best measure when the database will be reused. `one_off_total_seconds` adds database construction and is more relevant to a single analysis.


In [ ]:
search_runtime = (
    runtime_df[runtime_df.stage == "search_and_output"]
    .groupby(["chain_type", "tool", "setting"], as_index=False)
    .agg(
        median_search_seconds=("wall_seconds", "median"),
        min_search_seconds=("wall_seconds", "min"),
        max_search_seconds=("wall_seconds", "max"),
        peak_memory_mb=("peak_memory_mb", "max"),
    )
)
build_runtime = runtime_df[runtime_df.stage == "database_build"][["chain_type", "tool", "wall_seconds", "peak_memory_mb"]].rename(
    columns={"wall_seconds": "database_build_seconds", "peak_memory_mb": "database_build_peak_mb"}
)
runtime_summary_df = search_runtime.merge(build_runtime, on=["chain_type", "tool"], how="left")
runtime_summary_df["one_off_total_seconds"] = runtime_summary_df.database_build_seconds + runtime_summary_df.median_search_seconds
display(runtime_summary_df.sort_values(["chain_type", "median_search_seconds"]))

comparison_df = accuracy_df.merge(runtime_summary_df, on=["chain_type", "tool", "setting"], how="left")
display(comparison_df.sort_values(["identity_threshold", "chain_type", "f1"], ascending=[True, True, False]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

runtime_plot = runtime_summary_df.copy()
runtime_plot["label"] = runtime_plot.tool + " " + runtime_plot.setting
labels = runtime_plot.label.unique().tolist()
x = np.arange(len(labels))
width = 0.35
present_chains = sorted(runtime_plot.chain_type.unique().tolist())
offsets = np.linspace(-width / 2, width / 2, num=len(present_chains)) if len(present_chains) > 1 else [0]
for offset, chain_type in zip(offsets, present_chains):
    chain_data = runtime_plot[runtime_plot.chain_type == chain_type].set_index("label").reindex(labels)
    axes[0].bar(x + offset, chain_data.median_search_seconds, width, label=f"chain {chain_type}")
axes[0].set_xticks(x, labels, rotation=30, ha="right")
axes[0].set_ylabel("median search + output seconds")
axes[0].set_title("Runtime comparison")
axes[0].legend()

plot_threshold = 70.0 if 70.0 in IDENTITY_THRESHOLDS else IDENTITY_THRESHOLDS[0]
tradeoff = comparison_df[comparison_df.identity_threshold == plot_threshold].dropna(subset=["median_search_seconds", "recall"]).copy()
tradeoff["label"] = tradeoff.tool + " " + tradeoff.setting + " chain " + tradeoff.chain_type
for row in tradeoff.itertuples(index=False):
    axes[1].scatter(row.median_search_seconds, row.recall, s=70)
    axes[1].annotate(row.label, (row.median_search_seconds, row.recall), xytext=(4, 4), textcoords="offset points", fontsize=8)
axes[1].set_xlabel("median search + output seconds")
axes[1].set_ylabel("recall")
axes[1].set_ylim(0, 1.03)
axes[1].set_title(f"Runtime–recall tradeoff at {plot_threshold:g}% identity")
axes[1].grid(alpha=0.25)

plt.tight_layout()
figure_path = OUTPUT_DIR / "BLAST_vs_MMseqs2_runtime_accuracy.png"
plt.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()


## 12. How to decide which tool is better

Do not choose from runtime alone. Look for the best point on the runtime–recall tradeoff:

- If MMseqs2 has approximately the same recall and F1 but substantially lower runtime, MMseqs2 is the better high-throughput choice.
- If BLAST recovers noticeably more true pairs at the identity level you care about, BLAST may be preferable when sensitivity is more important than speed.
- A higher MMseqs2 `-s` value is expected to trade additional runtime for sensitivity.
- Results on a few hundred highly similar TCR domains are specific to this dataset; they should not be presented as a universal benchmark of either program.
- BLAST and MMseqs2 bit scores are generated by different implementations and are not compared directly here. The comparison uses recovered sequence pairs against the same independent ground truth.

References: [ANARCI paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC4708101/), [NCBI BLAST+ manual](https://www.ncbi.nlm.nih.gov/books/NBK279684/), and [MMseqs2 documentation](https://github.com/soedinglab/mmseqs2).


## 13. Export all results

The archive contains the ANARCI audit, alpha/beta FASTAs, raw timing records, standardized search results, exact ground truth, accuracy metrics, summary table and figure.


In [ ]:
EXPORT_DIR = OUTPUT_DIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
input_df.to_csv(EXPORT_DIR / "input_sequence_audit.csv", index=False)
failure_df.to_csv(EXPORT_DIR / "ANARCI_failures.csv", index=False)
cdr_df.to_csv(EXPORT_DIR / "TCR_CDR_IMGT_results.csv", index=False)
benchmark_df.to_csv(EXPORT_DIR / "benchmark_sequence_manifest.csv", index=False)
runtime_df.to_csv(EXPORT_DIR / "runtime_all_runs.csv", index=False)
runtime_summary_df.to_csv(EXPORT_DIR / "runtime_summary.csv", index=False)
accuracy_df.to_csv(EXPORT_DIR / "accuracy_precision_recall_f1.csv", index=False)
comparison_df.to_csv(EXPORT_DIR / "BLAST_vs_MMseqs2_comparison.csv", index=False)
truth_runtime_df.to_csv(EXPORT_DIR / "exact_ground_truth_runtime.csv", index=False)
pd.DataFrame([{
    "run_id": RUN_ID, "input_records": len(input_df), "anarci_candidates": len(candidate_rows),
    "accepted_domains": len(cdr_df), "anarci_wall_seconds": anarci_wall_seconds,
    "threads": THREADS, "repeats": N_REPEATS, "minimum_coverage": MIN_COVERAGE,
}]).to_csv(EXPORT_DIR / "run_metadata.csv", index=False)

for chain_type, truth_df in truth_tables.items():
    truth_df.to_csv(EXPORT_DIR / f"exact_ground_truth_chain_{chain_type}.csv", index=False)
for (chain_type, tool, setting), frame in standardized_results.items():
    safe_setting = setting.replace("=", "_").replace(".", "p")
    frame.to_csv(EXPORT_DIR / f"standardized_{chain_type}_{tool}_{safe_setting}.csv", index=False)
for fasta_path in benchmark_fastas.values():
    shutil.copy2(fasta_path, EXPORT_DIR / fasta_path.name)
shutil.copy2(figure_path, EXPORT_DIR / figure_path.name)

archive_base = Path("/content") / f"ANARCI_BLAST_MMseqs2_benchmark_results_{RUN_ID}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=EXPORT_DIR))
print("Created:", archive_path)
files.download(str(archive_path))
